In [1]:
# Install Google's Gemini SDK
%pip install -q -U google-genai

# Import required libraries
from google import genai
from kaggle_secrets import UserSecretsClient

# Access your Kaggle secret
secrets = UserSecretsClient()
api_key = secrets.get_secret("DR Report")

# Create Gemini client
client = genai.Client(api_key=api_key)

print("Gemini API connection configured successfully!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-aiplatform 2.7.0 requires google-genai<3.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 3.4.0 which is incompatible.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.4.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Gemi

In [2]:
# Select the Gemini model
MODEL_NAME = "gemini-3.6-flash"

# Send a test prompt to Gemini
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Explain what an LLM is in simple words in 3 sentences."
)

# Display the generated response
print(response.text)

A Large Language Model (LLM) is a smart computer program that has learned by reading massive amounts of text. Because it understands patterns in how words fit together, it can answer questions, write stories, and translate languages. Think of it as a supercharged version of your phone's predictive text that can hold a full conversation with you.


In [3]:
# Sample output from our CNN model
sample_prediction = {
    "predicted_class": "Moderate",
    "confidence": 0.76,
    "probabilities": {
        "No DR": 0.04,
        "Mild": 0.10,
        "Moderate": 0.76,
        "Severe": 0.07,
        "Proliferative DR": 0.03
    }
}

print(sample_prediction)

{'predicted_class': 'Moderate', 'confidence': 0.76, 'probabilities': {'No DR': 0.04, 'Mild': 0.1, 'Moderate': 0.76, 'Severe': 0.07, 'Proliferative DR': 0.03}}


In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".keras"):
            print(os.path.join(root, file))

/kaggle/input/models/hemasakshis/dr-classification-model-/tensorflow2/default/1/final_dr_model.keras


In [5]:
import tensorflow as tf

MODEL_PATH = "/kaggle/input/models/hemasakshis/dr-classification-model-/tensorflow2/default/1/final_dr_model.keras"

cnn_model = tf.keras.models.load_model(MODEL_PATH)

print("CNN model loaded successfully!")
cnn_model.summary()

I0000 00:00:1794879447.760544      16 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794879462.449083      16 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1794879467.360079      16 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


CNN model loaded successfully!


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │        10,245 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,574,545 (204.37 MB)

 Trainable params: 14,988,293 (57.18 MB)

 Non-trainable params: 8,609,664 (32.84 MB)

 Optimizer params: 29,976,588 (114.35 MB)

In [6]:
import os

DATASET_PATH = "/kaggle/input/competitions/aptos2019-blindness-detection"

print(os.listdir(DATASET_PATH))

['sample_submission.csv', 'train_images', 'train.csv', 'test.csv', 'test_images']


In [7]:
import pandas as pd
import os

# Load the dataset labels
csv_path = os.path.join(DATASET_PATH, "train.csv")
df = pd.read_csv(csv_path)

# Select the first retinal image
sample_image_name = df.iloc[0]["id_code"]
actual_label = df.iloc[0]["diagnosis"]

# Get its image path
image_path = os.path.join(
    DATASET_PATH,
    "train_images",
    sample_image_name + ".png"
)

print("Image name:", sample_image_name)
print("Actual DR grade:", actual_label)
print("Image path:", image_path)

Image name: 000c1434d8d7
Actual DR grade: 2
Image path: /kaggle/input/competitions/aptos2019-blindness-detection/train_images/000c1434d8d7.png


In [8]:
import tensorflow as tf

# Read the retinal image
image = tf.io.read_file(image_path)

# Decode the image into RGB format
image = tf.image.decode_png(image, channels=3)

# Resize to the model's expected input size
image = tf.image.resize(image, [224, 224])

# Apply ResNet50 preprocessing
image = tf.keras.applications.resnet50.preprocess_input(image)

# Add batch dimension
image = tf.expand_dims(image, axis=0)

print("Prepared image shape:", image.shape)
print("Image preprocessing completed!")

Prepared image shape: (1, 224, 224, 3)
Image preprocessing completed!


In [9]:
# Get predictions from the CNN
predictions = cnn_model.predict(image)

# Get the class with the highest probability
predicted_class = predictions.argmax(axis=1)[0]

# Get the confidence score
confidence = predictions[0][predicted_class]

# DR class names
class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR"
]

print("Predicted DR Grade:", class_names[predicted_class])
print("Confidence:", round(float(confidence) * 100, 2), "%")

print("\nClass-wise probabilities:")

for i, probability in enumerate(predictions[0]):
    print(class_names[i], ":", round(float(probability) * 100, 2), "%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Predicted DR Grade: Moderate
Confidence: 69.47 %

Class-wise probabilities:
No DR : 0.02 %
Mild : 2.55 %
Moderate : 69.47 %
Severe : 4.39 %
Proliferative DR : 23.56 %


In [10]:
print("Actual DR Grade:", class_names[int(actual_label)])
print("Predicted DR Grade:", class_names[predicted_class])

if int(actual_label) == predicted_class:
    print("Result: Correct Prediction")
else:
    print("Result: Incorrect Prediction")

Actual DR Grade: Moderate
Predicted DR Grade: Moderate
Result: Correct Prediction


In [11]:
# Convert CNN prediction into structured data
cnn_output = {
    "predicted_class": class_names[predicted_class],
    "confidence": float(confidence),
    "probabilities": {
        class_names[i]: float(predictions[0][i])
        for i in range(len(class_names))
    }
}

print(cnn_output)

{'predicted_class': 'Moderate', 'confidence': 0.6947429180145264, 'probabilities': {'No DR': 0.00021847766765858978, 'Mild': 0.025516953319311142, 'Moderate': 0.6947429180145264, 'Severe': 0.04388175532221794, 'Proliferative DR': 0.23563992977142334}}
